In [1]:
import sys
import os
import pandas
import xarray
import rioxarray
import contextily as cx
import rasterio
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import geopandas as gpd
import numba
import numpy as np
import time


In [2]:
path_to_data = r'/Users/jonathanelsey/Projects/Imago/Data/cloudProb_tif/cloudProb/cloudprob_uk_2024_NZ26.tif'
full_tiff = (
    rioxarray.open_rasterio(path_to_data, chunks=True)
    .compute()
)

## Threading backend issues
Apple, in their infinite wisdom, removed OpenMP support from their Clang that they use (they of course want us to use their proprietary threading engine). And Intel's TBB won't work either as it is not designed to run on Apple silicon.
So we are stuck with `workqueue` for Numba's threading backend without doing a complete rebuild of the virtual environment.

This is *bad*. We can't use `workqueue` with `dask`, as this will be interpreted as nested multithreading, and `workqueue` is *not* thread safe. 

SO - will do initial testing of the averaging kernel using `workqueue`, then move over to a HPC or my own Linux machine for testing with halo exchange + all tiles. 

*N.B. - could consider UTM to run as a lightweight VM?*


In [3]:

import os


from numba import config
config.THREADING_LAYER = 'workqueue'   # tbb and omp dont work 

@numba.njit(parallel=False)
def foo(a, b):
    return a + b
    
x = np.arange(10.)
y = x.copy()
foo(x, y)
import numba
print(numba.threading_layer())


ValueError: Threading layer is not initialized.

In [ ]:
# We're going to make a bunch of plots, so here's something that makes plots.
def plot_sp(
    sp,
    ax=None,
    remove_axis=True,
    alpha=0.75,
    basemap_alpha=1,
    basemap_source=cx.providers.CartoDB.Positron,
    vmin=None,
    vmax=None,
    cmap=None,
    figsize=None,
    fig_width=10
):
    if ax is None:
        if figsize is None:
            # Calculate aspect ratio from raster bounds
            if hasattr(sp, 'rio') and hasattr(sp.rio, 'bounds'):
                bounds = sp.rio.bounds()
                width = bounds[2] - bounds[0]
                height = bounds[3] - bounds[1]
                aspect_ratio = width / height
            else:
                # Fallback to shape if rio not available
                height, width = sp.shape[-2:]
                aspect_ratio = width / height

            fig_height = fig_width / aspect_ratio
            figsize = (fig_width, fig_height)

        f, ax = plt.subplots(1, figsize=figsize)

    sp.plot.imshow(ax=ax, alpha=alpha, vmin=vmin, vmax=vmax, cmap=cmap)
    cx.add_basemap(ax, alpha=basemap_alpha, crs=sp.rio.crs, source=basemap_source, zorder=-1)

    if remove_axis:
        ax.set_axis_off()

    return ax

#### Scipy implementation of the convolution filter - this is what we had before

In [ ]:
from scipy.ndimage import convolve

def weighted_spatial_lag_conv(prob_array, acq_array, kernel_size=3):
    """
    Acquisition-weighted spatial lag using convolution.
    Matches: np.average(neighbor_probs, weights=neighbor_acqs)
    """
    prob = prob_array.values if hasattr(prob_array, 'values') else prob_array
    acq = acq_array.values if hasattr(acq_array, 'values') else acq_array

    # Create kernel (all neighbors, exclude center)
    kernel = np.ones((kernel_size, kernel_size))
    kernel[kernel_size // 2, kernel_size // 2] = 0

    # Handle NaNs - treat as 0 for convolution
    prob_filled = np.nan_to_num(prob, nan=0.0)
    acq_filled = np.nan_to_num(acq, nan=0.0)

    # Acquisition-weighted sum: Σ(prob_i * acq_i)
    weighted_prob_sum = convolve(prob_filled * acq_filled, kernel, mode='constant')

    # Sum of acquisition weights: Σ(acq_i)
    acq_sum = convolve(acq_filled, kernel, mode='constant')

    # Weighted average: Σ(prob_i * acq_i) / Σ(acq_i)
    weighted_lag = np.divide(
        weighted_prob_sum,
        acq_sum,
        where=acq_sum > 0,
        out=np.full_like(weighted_prob_sum, np.nan)
    )

    result = prob_array.copy(data=weighted_lag)
    return result

In [ ]:
%%time
wy_convolved = weighted_spatial_lag_conv(
    full_tiff.sel(band=1),
    full_tiff.sel(band=2),
    kernel_size=21 # 21x21 window = 441 queen neighbors
)

#### Walltime is ~4 seconds. This is on its own almost two orders of magnitude faster than Dani's initial implementation with PySAL, but I think we can do better. 

#### Numba implementation of the convolution filter - with some profiling. Strictly we should use timeit here for multiple measurements, but we don't care about ~5-10% noise here, we're looking for large speedups.

In [ ]:
# Let's try implementing with Numba, rather than using scipy's convolve. 
# This should reduce the runtime a bit since we avoid all the Python overhead in the setup
# It is also easily parallelisable with `prange`

fastmath=False

@numba.njit(parallel=False, fastmath=fastmath)
def weighted_spatial_lag_numba(prob, acq, kernel_size=3):
    nrows, ncols = prob.shape
    result = np.full_like(prob, np.nan)
    offset = kernel_size // 2

    for i in numba.prange(nrows):
        for j in range(ncols):
            w_sum = 0.0
            a_sum = 0.0
            for ki in range(-offset, offset+1):
                for kj in range(-offset, offset+1):
                    if ki == 0 and kj == 0:
                        continue
                    ii, jj = i+ki, j+kj
                    if 0 <= ii < nrows and 0 <= jj < ncols:
                        w_sum += prob[ii, jj] * acq[ii, jj]
                        a_sum += acq[ii, jj]
            if a_sum > 0:
                result[i, j] = w_sum / a_sum
    return result



#### First pass - will be slower since we have the overhead of compiling the function at runtime


In [ ]:
%%time 
numba.set_num_threads(1)  # force 1 thread for now 

result = weighted_spatial_lag_numba(
    full_tiff.sel(band=1).data,
    full_tiff.sel(band=2).data,
    kernel_size=21) # 21x21 window = 441 queen neighbors%%time


#### Second pass - no compilation overhead.

In [ ]:
%%time
start = time.time()
result = weighted_spatial_lag_numba(
    full_tiff.sel(band=1).data,
    full_tiff.sel(band=2).data,
    kernel_size=21) # 21x21 window = 441 queen neighbors
end = time.time()
ref_time = end - start
print('Time taken (walltime) = ', ref_time)

#### Already we have reduced the runtime by ~50%, without even including any parallelisation. Let's try with different thread counts.

In [ ]:
thread_nums = [1, 2, 4, 6, 8, 10, 12]
import timeit

# kwargs so that we can call func_to_benchmark with the chunksize later
def benchmark_func(func_to_benchmark, ref_time, **kwargs):
    for thread_count in thread_nums:
        numba.set_num_threads(thread_count)
        start = time.time()
        result = func_to_benchmark(
            full_tiff.sel(band=1).data,
            full_tiff.sel(band=2).data,
            kernel_size=21,
            **kwargs
        ) 
        end = time.time()
        parallel_time = end - start
        print(f'Total time taken for {thread_count} threads is {parallel_time:2f} seconds')
        print(f'Percent speedup = {100 * ref_time/parallel_time}%, ideal speedup is {100 * thread_count}%')

benchmark_func(weighted_spatial_lag_numba, ref_time)

#### This is some pretty great scaling - for 4/6 threads we get effectively ideal speedup. We may be able to do a *little* better by flattening the outer loop. This won't make much difference for a serial case most likely, but might do some work for a parallel loop

In [ ]:

@numba.njit(parallel=False, fastmath=fastmath)
def weighted_spatial_lag_flat(prob, acq, kernel_size=21):
    nrows, ncols = prob.shape
    out = np.full((nrows, ncols), np.nan, dtype=np.float32)
    pad = kernel_size // 2

    for idx in numba.prange(nrows * ncols):
        i = idx // ncols
        j = idx % ncols
        w_sum = 0.0
        a_sum = 0.0
        for di in range(-pad, pad + 1):
            for dj in range(-pad, pad + 1):
                if di == 0 and dj == 0:
                    continue
                ii = i + di
                jj = j + dj
                if 0 <= ii < nrows and 0 <= jj < ncols:
                    w_sum += prob[ii, jj] * acq[ii, jj]
                    a_sum += acq[ii, jj]
        if a_sum > 0:
            out[i, j] = w_sum / a_sum
    return out


In [ ]:
%%time 
# First pass to avoid compilation overhead 

result = weighted_spatial_lag_flat(
    full_tiff.sel(band=1).data,
    full_tiff.sel(band=2).data,
    kernel_size=21) # 21x21 window = 441 queen neighbors%%time


In [ ]:
benchmark_func(weighted_spatial_lag_flat, ref_time)

#### Not much of a change here. Another thing we can try is cache blocking, where we tile the loops so that the data can fit into cache more nicely.


In [ ]:

@numba.njit(parallel=False, fastmath=fastmath)
def weighted_spatial_lag_tiled(prob, acq, kernel_size=21, chunksize=128):
    nrows, ncols = prob.shape
    out = np.full((nrows, ncols), np.nan, dtype=np.float32)
    pad = kernel_size // 2
    block_h = chunksize
    block_w = chunksize
    print('Chunksize = ', chunksize)
    n_blocks_i = (nrows + block_h - 1) // block_h

    # parallelize over rows again
    for bi in numba.prange(n_blocks_i):
        i0 = bi * block_h
        i1 = min(i0 + block_h, nrows)

        for bj in range(0, ncols, block_w):  
            j0 = bj
            j1 = min(j0 + block_w, ncols)

            for i in range(i0, i1):
                for j in range(j0, j1):
                    w_sum = 0.0
                    a_sum = 0.0
                    for di in range(-pad, pad + 1):
                        for dj in range(-pad, pad + 1):
                            if di == 0 and dj == 0:
                                continue
                            ii = i + di
                            jj = j + dj
                            if 0 <= ii < nrows and 0 <= jj < ncols:
                                w_sum += prob[ii, jj] * acq[ii, jj]
                                a_sum += acq[ii, jj]
                    if a_sum > 0:
                        out[i, j] = w_sum / a_sum
    return out


In [ ]:
# again compile 
result = weighted_spatial_lag_flat(
    full_tiff.sel(band=1).data,
    full_tiff.sel(band=2).data,
    kernel_size=21) # 21x21 window = 441 queen neighbors%%time

thread_nums = [1, 2, 4]
chunksizes = [16, 32, 64, 128, 256, 512, 1028]

for size in chunksizes:
    benchmark_func(weighted_spatial_lag_tiled, ref_time, chunksize=size)


#### Chunksize of 128 seems optimal. But not much difference compared to not chunking. Likely some optimisations in LLVM anyway so chunking may be unnecessary. The main thought behind this is that it will make things much better if we switch to GPU down the line.

#### Worse performance with 4 threads at large chunksize since we likely have too large chunks compared to the number of threads, so we are losing CPU utilisation. 

#### Likely that flattening the loop *would* make a difference in this case, but likely too small to worry about.

### Convolving over many tiles
Effectively we have a halo exchange - each tile needs to know some information about its neighbours. We want to operate on the tiles in parallel. We can do some of this with threading, but for true scalability I think we need a combination of threads + workers (kind of like MPI + OpenMP). 

If we implement any additional processing, this should fit the same pattern. If some of that processing is expensive, it may be possible to port some of this to GPU, but I don't think is necessary for this convolution even with big data since it is on the order of milliseconds to process one tile out of 800.

What I think we want to do is:
- Load in every tile as one dataset, lazily evaluated. To do this, use `chunks` parameter to `open_mfdataset` (see below)
- For each tile, we effectively need to construct a new tile (in memory) of size `(X + N, Y + N)` where `N` is (`(kernel_size - 1) / 2`), then perform the convolution. 
- Then, clip the tile to the original bounds.
- This will require information about all 9 adjacent tiles to construct at a time.
- I think we can use `dask.array.map_overlap` (https://docs.dask.org/en/latest/generated/dask.array.map_overlap.html) to handle the halo exchange for us. Then, we just need to use `xarray.open_mfdataset` (https://docs.xarray.dev/en/stable/generated/xarray.open_mfdataset.html) to load in all of the tiles, construct a single array and then parallelise over the tiles with `Dask`.
- We can see from the above that ~4 threads gives effectively ideal scaling, so we can use threads in addition to workers to keep the graph nice and compact while still having max CPU utilisation.
- The bigger job honestly is actually constructing the dataset of all tiles. After this, it is trivial to just loop through the tiles in parallel with Dask. We will of course need to handle the edges of the edge tiles, but since these are in the sea we don't really care as they won't affect things at LSOA level.

#### Let's try this with a small subset of tiles. Pick a single tile, then make sure we get the 9 tiles around it. 

#### First up, let's just load in one tile and see what information it holds.

In [ ]:
from xarray import open_dataset

In [ ]:
data_dir = '/Users/jonathanelsey/Projects/Imago/Data/cloudProb_netcdf/cloudProb_nc/2024'
testdata = open_dataset(f'{data_dir}/cloudprob_uk_2024_NC06.nc')

In [ ]:
x_chunk = len(testdata.x)
y_chunk = len(testdata.y)

In [ ]:
all_data_lazy = xarray.open_mfdataset(
    f"{data_dir}/cloudprob_uk_2024_*.nc",
    combine="by_coords",  # merges based on x/y coordinates
    chunks={"x": x_chunk, "y": y_chunk},  # make sure we retain the tiles as chunks, they're all the same size
    engine='netcdf4'
)

In [ ]:
all_data_lazy

In [ ]:
# separate Dask arrays for probability and acquisitions
prob_da = all_data_lazy['cloud_prob'].isel(year=0)
acq = all_data_lazy['valid_count'].isel(year=0)
prob = all_data_lazy["cloud_prob"].isel(year=0).data 
acq  = all_data_lazy["valid_count"].isel(year=0).data

In [ ]:
all_data_lazy['cloud_prob'].isel(year=0).data


In [ ]:
#### Need to set Numba threading layer to TBB or OpenMP, not workqueue here. This part won't run on Mac, so move to Linux.

#### Let's try something simple - loop over each tile with the halo exchange, and submit each to the client as a separate task.

In [ ]:

import os
import re
import numpy as np
import xarray as xr
from dask.distributed import Client, LocalCluster
from dask import delayed


cluster = LocalCluster(n_workers=4, threads_per_worker=2, memory_limit="8GB")
client = Client(cluster)

data_dir = "/Users/jonathanelsey/Projects/Imago/Data/cloudProb_netcdf/cloudProb_nc"
files = sorted([f for f in os.listdir(data_dir) if f.endswith(".nc")])

print('Running halo exchange task')
tile_pat = re.compile(r".*_(?P<tile>[A-Z]{2}\d{2})\.nc$")

def add_tile_id(ds, filename):
    m = tile_pat.match(os.path.basename(filename))
    tile_id = m.group("tile")
    return ds.assign_coords(tile_id=tile_id)

datasets = []
for f in files:
    ds = xr.open_dataset(os.path.join(data_dir, f), chunks={"y": 1024, "x": 1024})
    datasets.append(add_tile_id(ds, f))


kernel_size = 21
halo = kernel_size // 2
out_dir = os.path.join(data_dir, "tiles_out")
os.makedirs(out_dir, exist_ok=True)

def process_tile(prob_tile, acq_tile, kernel_size, halo):
    prob_np = prob_tile.values
    acq_np = acq_tile.values

    # halo exchange
    prob_padded = np.pad(prob_np, halo, mode="constant")
    acq_padded = np.pad(acq_np, halo, mode="constant")

    result = weighted_spatial_lag_numba(prob_padded, acq_padded, kernel_size)
    # clip to original shape
    result = result[halo:-halo, halo:-halo]
    return result

futures = []
for ds in datasets:
    tile_id = ds.tile_id.item()
    prob_tile = ds["cloud_prob"].isel(year=0)
    acq_tile = ds["valid_count"].isel(year=0)

    fut = client.submit(process_tile, prob_tile, acq_tile, kernel_size, halo)
    futures.append((tile_id, fut))


for tile_id, fut in futures:
    result_np = fut.result()
    y_coords = prob_tile["y"].values
    x_coords = prob_tile["x"].values
    lagged_da = xr.DataArray(result_np, dims=("y", "x"), coords={"y": y_coords, "x": x_coords}, name="cloudprob_weighted")

    fname = os.path.join(out_dir, f"cloudprob_weighted_{tile_id}.nc")
    lagged_da.to_netcdf(fname, encoding={"cloudprob_weighted": {"dtype": "float32", "zlib": True, "complevel": 4}})
